# KLD-Minimization (Randomized) — All Tasks, Multiple LAMBDA

This merged notebook runs the KL-divergence-minimization debiasing experiment for **all four tasks**
(Toxicity, Sentiment, Sarcasm, Hate-speech) across **two LAMBDA values (3 and 5)** in a single run.

For every (task, LAMBDA) pair it:
1. Loads the task dataset and pretrained model.
2. Fine-tunes a fresh model (KLD-minimization + randomized-side CE loss).
3. Evaluates on the test set for the male and female framing.
4. Computes bootstrap accuracy CIs and fairness metrics (SPD, EOD).
5. Saves per-run predictions and appends a row to a global results summary.

**How to use:** set `LAMBDAS`, `TASKS`, and training hyperparameters in the *Configuration* cell,
then run all cells. The final cell prints/saves the combined `all_results` table.


## Configuration

In [ ]:
# --- Global experiment configuration ------------------------------------

# Which lambda values to run
LAMBDAS = [1, 3, 5]

# Training controls (applied to every task/lambda run)
FINETUNE   = True     # True -> fine-tune fresh each run; False -> load saved checkpoint & only run inference
EPOCHS     = 15
LR         = 1e-4
BATCH_SIZE = 4
MAX_LENGTH = 512
SAVE_MODEL = True

# Per-task settings. Each task is fully described here so the pipeline below is generic.
#   data       : CSV with columns [male, female, _original_label, test]
#   model_path : HuggingFace model id used as the pretrained backbone
#   ckpt       : base checkpoint filename (LAMBDA is appended automatically)
#   result     : base result-CSV filename (LAMBDA is appended automatically)
#   balance    : if True, oversample minority class in train split (sarcasm did this)
TASKS = {
    "toxicity": {
        "data":       "train_test_toxicity_data.csv",
        "model_path": "FredZhang7/one-for-all-toxicity-v3",
        "ckpt":       "Toxicity_KLDM_Randomized",
        "result":     "revision_kldm_randomized_toxicity_result",
        "balance":    False,
    },
    "senti": {
        "data":       "train_test_senti_data.csv",
        "model_path": "ka05ar/banglabert-sentiment",
        "ckpt":       "Senti_KLDM_Randomized",
        "result":     "revision_kldm_randomized_senti_result",
        "balance":    False,
    },
    "sarcasm": {
        "data":       "train_test_sarcasm_data_mod.csv",
        "model_path": "raquiba/sarcasm-detection-BanglaSARC",
        "ckpt":       "Sarcasm_KLDM_Randomized",
        "result":     "revision_kldm_randomized_sarcasm_result",
        "balance":    True,
    },
    "hatespeech": {
        "data":       "train_test_hatespeech_data.csv",
        "model_path": "Hate-speech-CNERG/bengali-abusive-MuRIL",
        "ckpt":       "HateSpeech_KLDM_Randomized",
        "result":     "revision_kldm_randomized_hatespeech_result",
        "balance":    False,
    },
}

# Choose which tasks to run (edit to a subset if you only want some)
RUN_TASKS = list(TASKS.keys())

print("Will run tasks:", RUN_TASKS)
print("Lambda values :", LAMBDAS)


## Imports

In [ ]:
import random, time, warnings
from collections import OrderedDict, namedtuple
from itertools import product
from datetime import datetime

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForMaskedLM, get_linear_schedule_with_warmup
from tqdm import tqdm
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Reusable Components
The dataset class, model, training-stat tracker, trainer, prediction and metric functions are
defined once here and reused for every (task, LAMBDA) combination.

In [ ]:
# ---- Dataset -----------------------------------------------------------
class BertDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        super().__init__()
        self.train_csv = data
        self.tokenizer = tokenizer
        self.target = self.train_csv.iloc[:, 2]
        self.max_length = max_length

    def __len__(self):
        return len(self.train_csv)

    def __getitem__(self, index):
        text1 = self.train_csv.iloc[index, 0]  # male text
        text2 = self.train_csv.iloc[index, 1]  # female text
        inputs = self.tokenizer.batch_encode_plus(
            [text1, text2],
            pad_to_max_length=True,
            add_special_tokens=True,
            return_attention_mask=True,
            max_length=self.max_length,
            truncation=True,
        )
        ids = inputs["input_ids"]
        token_type_ids = inputs["token_type_ids"]
        masks = inputs["attention_mask"]
        return (
            torch.tensor(ids[0], dtype=torch.long),
            torch.tensor(token_type_ids[0], dtype=torch.long),
            torch.tensor(masks[0], dtype=torch.long),
            torch.tensor(ids[1], dtype=torch.long),
            torch.tensor(token_type_ids[1], dtype=torch.long),
            torch.tensor(masks[1], dtype=torch.long),
            torch.tensor(self.train_csv.iloc[index, 2], dtype=torch.long),
        )


In [ ]:
# ---- Model -------------------------------------------------------------
class BERT(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.bert_model = model
        self.dropout = nn.Dropout(0.2)
        self.out = nn.Linear(768, 2)

    def forward(self, ids1, mask1, token_type_ids1, ids2, mask2, token_type_ids2, mode):
        if mode == "training":
            ids = torch.vstack((ids1, ids2))
            mask = torch.vstack((mask1, mask2))
            token_type_ids = torch.vstack((token_type_ids1, token_type_ids2))
            o2 = self.bert_model(ids, mask, token_type_ids).hidden_states[0]
            o2 = o2.max(dim=1).values
            male_embedding, female_embedding = torch.vsplit(o2, 2)
            out_male = self.out(self.dropout(male_embedding))
            out_female = self.out(self.dropout(female_embedding))
            # randomization: supervise a randomly-chosen side each step
            c = np.random.randint(0, 2)
            output = out_male if c == 0 else out_female
            return output, F.softmax(out_male, dim=-1), F.softmax(out_female, dim=-1)

        if mode == "inference":
            o2 = self.bert_model(ids1, mask1, token_type_ids1).hidden_states[0]
            oe = o2.max(dim=1).values
            out = self.out(self.dropout(oe))
            return out, oe


def load_transformer_based_model(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForMaskedLM.from_pretrained(model_path, output_hidden_states=True)
    return tokenizer, model


def build_model(model_path):
    """Load backbone, add special tokens, wrap in BERT head + DataParallel."""
    tokenizer, b_model = load_transformer_based_model(model_path)
    tokenizer.add_tokens(["<Name>", "<Gender>"])
    b_model.resize_token_embeddings(len(tokenizer))
    net = nn.DataParallel(BERT(b_model)).to(device)
    return tokenizer, net


In [ ]:
# ---- Training-stat tracker --------------------------------------------
class RunBuilder:
    @staticmethod
    def get_runs(params):
        Run = namedtuple("Run", params.keys())
        return [Run(*v) for v in product(*params.values())]


class RunManager:
    def __init__(self):
        self.epoch_count = 0
        self.run_count = 0
        self.run_data = []
        self.network = self.loader = self.validation_loader = None

    def begin_run(self, run, network, loader, validation_loader):
        self.run_start_time = time.time()
        self.run_params = run
        self.run_count += 1
        self.network = network
        self.loader = loader
        self.validation_loader = validation_loader

    def end_run(self):
        self.epoch_count = 0

    def begin_epoch(self):
        self.epoch_start_time = time.time()
        self.epoch_count += 1
        self.epoch_ce_loss = self.epoch_kl_loss = self.epoch_loss = 0
        self.epoch_num_correct = 0
        self.epoch_valid_ce_loss = self.epoch_valid_kl_loss = self.epoch_valid_loss = 0
        self.epoch_num_valid_correct = 0

    def end_epoch(self):
        epoch_duration = time.time() - self.epoch_start_time
        run_duration = time.time() - self.run_start_time
        n_tr = len(self.loader.dataset)
        n_va = len(self.validation_loader.dataset)

        results = OrderedDict()
        results["run"] = self.run_count
        results["epoch"] = self.epoch_count
        results["ce_loss"] = self.epoch_ce_loss / n_tr
        results["kl_loss"] = self.epoch_kl_loss / n_tr
        results["loss"] = self.epoch_loss / n_tr
        results["accuracy"] = self.epoch_num_correct / n_tr
        results["validation ce loss"] = self.epoch_valid_ce_loss / n_va
        results["validation kl loss"] = self.epoch_valid_kl_loss / n_va
        results["validation loss"] = self.epoch_valid_loss / n_va
        results["validation accuracy"] = self.epoch_num_valid_correct / n_va
        results["epoch duration (minutes)"] = epoch_duration / 60
        results["run duration (minutes)"] = run_duration / 60
        for k, v in self.run_params._asdict().items():
            results[k] = v
        self.run_data.append(results)
        df = pd.DataFrame.from_dict(self.run_data, orient="columns")
        clear_output(wait=True)
        display(df)

    def track_loss(self, ce_loss, kl_loss, loss):
        self.epoch_ce_loss += ce_loss.item()
        self.epoch_kl_loss += kl_loss.item()
        self.epoch_loss += loss.item()

    def track_num_correct(self, preds, labels):
        self.epoch_num_correct += self._get_num_correct(preds, labels)

    def track_validation_loss(self, ce_loss, kl_loss, loss):
        self.epoch_valid_ce_loss += ce_loss.item()
        self.epoch_valid_kl_loss += kl_loss.item()
        self.epoch_valid_loss += loss.item()

    def track_num_validation_correct(self, preds, labels):
        self.epoch_num_valid_correct += self._get_num_correct(preds, labels)

    @torch.no_grad()
    def _get_num_correct(self, preds, labels):
        return preds.max(1)[1].view(-1, 1).eq(labels).sum().item()

    def save(self, fileName):
        pd.DataFrame.from_dict(self.run_data, orient="columns").to_csv(rf"{fileName}.csv")


In [ ]:
# ---- Save / load helpers ----------------------------------------------
def save_model(network, optimizer, ckpt_path):
    torch.save({
        "model_state_dict": network.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, ckpt_path)
    print(f"Saved model and optimizer at {ckpt_path}")


def load_model(network, optimizer, PATH):
    checkpoint = torch.load(PATH, map_location=device)
    network.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        print(rf"Loaded model and optimizer from {PATH}")
    else:
        print(rf"Loaded model from {PATH}")


In [ ]:
# ---- Trainer -----------------------------------------------------------
def my_trainer(train_loader, validation_loader, network, ckpt_path, LAMBDA,
               LR=1e-5, epochs=5, fresh_training=True):
    params = OrderedDict(
        lr=[LR],
        batch_size=[BATCH_SIZE],
        device=["cuda" if torch.cuda.is_available() else "cpu"],
    )
    m = RunManager()
    for run in RunBuilder.get_runs(params):
        dev = torch.device(run.device)
        loss_fn = nn.CrossEntropyLoss()
        loss_fn_2 = nn.KLDivLoss(reduction="batchmean")
        total_steps = len(train_loader) * epochs
        optimizer = optim.Adam(network.parameters(), lr=run.lr)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
        if not fresh_training:
            load_model(network, optimizer, PATH=ckpt_path)
        m.begin_run(run, network, train_loader, validation_loader)

        for epoch in range(epochs):
            network.train(True)
            print(f"Epoch {epoch + 1} : ", end="\t")
            m.begin_epoch()

            for batch in tqdm(train_loader):
                (input_ids1, token_type_ids1, attention_masks1,
                 input_ids2, token_type_ids2, attention_masks2, labels) = [b.to(dev) if torch.is_tensor(b) else b for b in batch]
                labels = labels.type(torch.LongTensor).to(dev)
                copy_labels = labels
                preds, male_softmax, female_softmax = network(
                    input_ids1, attention_masks1, token_type_ids1,
                    input_ids2, attention_masks2, token_type_ids2, "training")
                labels = labels.view(-1, 1)
                loss1 = loss_fn(preds, copy_labels)
                loss2 = loss_fn_2(male_softmax.log(), female_softmax) + loss_fn_2(female_softmax.log(), male_softmax)
                loss = loss1 + LAMBDA * loss2
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                m.track_loss(loss1, loss2, loss)
                m.track_num_correct(preds, labels)

            network.eval()
            for batch in tqdm(validation_loader):
                (input_ids1, token_type_ids1, attention_masks1,
                 input_ids2, token_type_ids2, attention_masks2, labels) = [b.to(dev) if torch.is_tensor(b) else b for b in batch]
                labels = labels.type(torch.LongTensor).to(dev)
                copy_labels = labels
                with torch.no_grad():
                    preds, male_softmax, female_softmax = network(
                        input_ids1, attention_masks1, token_type_ids1,
                        input_ids2, attention_masks2, token_type_ids2, "training")
                    labels = labels.view(-1, 1)
                    loss1 = loss_fn(preds, copy_labels)
                    loss2 = loss_fn_2(male_softmax.log(), female_softmax) + loss_fn_2(female_softmax.log(), male_softmax)
                    loss = loss1 + LAMBDA * loss2
                    m.track_validation_loss(loss1, loss2, loss)
                    m.track_num_validation_correct(preds, labels)

            m.end_epoch()
            network.train(False)
            save_model(network, optimizer, ckpt_path)
        m.end_run()

    if SAVE_MODEL:
        save_model(network, optimizer, ckpt_path)
    s = datetime.now().strftime("%d-%m-%Y %I:%M:%p").replace(" ", "_").replace(":", "_")
    m.save(rf"results_{s}")


In [ ]:
# ---- Inference & metrics ----------------------------------------------
def get_prediction(data_loader, network, choice="male"):
    network = network.to(device)
    network.eval()
    l, emb = [], []
    for batch in tqdm(data_loader):
        (input_ids1, token_type_ids1, attention_masks1,
         input_ids2, token_type_ids2, attention_masks2, labels) = [b.to(device) if torch.is_tensor(b) else b for b in batch]
        with torch.no_grad():
            if choice == "male":
                preds, hidden_states = network(input_ids1, attention_masks1, token_type_ids1, None, None, None, "inference")
            else:
                preds, hidden_states = network(input_ids2, attention_masks2, token_type_ids2, None, None, None, "inference")
            preds = preds.max(1)[1].view(-1, 1)
            l.append(preds)
            emb.append(hidden_states)
    return l, torch.vstack(emb).cpu().numpy()


def bootstrap_ci(y_true, y_pred, B=500, seed=0):
    rng = np.random.RandomState(seed)
    N = len(y_true)
    original_acc = np.mean(y_pred == y_true)
    accs = []
    for _ in range(B):
        idx = rng.choice(N, N, replace=True)
        accs.append(np.mean(y_pred[idx] == y_true[idx]))
    lower, upper = np.percentile(accs, [2.5, 97.5])
    return original_acc, (lower, upper), np.mean(accs)


def calculate_spd(male_pred, female_pred):
    return np.abs(np.mean(male_pred == 1) - np.mean(female_pred == 1))


def calculate_eod(male_pred, female_pred, y_true):
    tpr_male = np.mean((male_pred == 1) & (y_true == 1))
    tpr_female = np.mean((female_pred == 1) & (y_true == 1))
    return np.abs(tpr_male - tpr_female)


def bootstrap_ci_spd_eod(male_pred, female_pred, y_true, n_iterations=500, ci=95, seed=0):
    rng = np.random.RandomState(seed)
    n = len(male_pred)
    spd_values = np.zeros(n_iterations)
    eod_values = np.zeros(n_iterations)
    for i in range(n_iterations):
        idx = rng.choice(n, size=n, replace=True)
        spd_values[i] = calculate_spd(male_pred[idx], female_pred[idx])
        eod_values[i] = calculate_eod(male_pred[idx], female_pred[idx], y_true[idx])
    lo = (100 - ci) / 2
    hi = 100 - lo
    spd = (np.mean(spd_values), np.percentile(spd_values, lo), np.percentile(spd_values, hi))
    eod = (np.mean(eod_values), np.percentile(eod_values, lo), np.percentile(eod_values, hi))
    return spd, eod


In [ ]:
# ---- Data prep --------------------------------------------------------
def prepare_splits(cfg):
    data = pd.read_csv(cfg["data"])
    temp = data[data["test"] != 1]
    test = data[data["test"] == 1]
    if cfg["balance"]:
        # sarcasm-style: small validation frac + minority-class oversampling
        validation = temp.groupby("_original_label", group_keys=False).apply(
            lambda x: x.sample(frac=0.02, random_state=1234))
        train = temp.drop(validation.index)
        diff = max(train._original_label.value_counts()) - min(train._original_label.value_counts())
        ex = train[train._original_label == 0].sample(n=diff, random_state=1234)
        train = pd.concat([train, ex]).reset_index(drop=True)
    else:
        validation = temp.groupby("_original_label", group_keys=False).apply(
            lambda x: x.sample(frac=0.2, random_state=101))
        train = temp.drop(validation.index)
    return train, validation, test


## Single-run driver
`run_experiment` executes one (task, LAMBDA) combination end-to-end and returns a summary dict.

In [ ]:
def run_experiment(task_name, LAMBDA):
    cfg = TASKS[task_name]
    tag = f"{task_name}_lambda{LAMBDA}"
    ckpt_path = f"{cfg['ckpt']}_lambda{LAMBDA}.pth"
    result_csv = f"{cfg['result']}_lambda{LAMBDA}.csv"

    print("=" * 90)
    print(f"RUN: task={task_name}  LAMBDA={LAMBDA}")
    print(f"  data={cfg['data']}  model={cfg['model_path']}")
    print(f"  ckpt={ckpt_path}  result={result_csv}")
    print("=" * 90)

    # data
    train, validation, test = prepare_splits(cfg)
    test = test.copy()

    # model
    tokenizer, network = build_model(cfg["model_path"])

    # loaders
    train_loader = DataLoader(BertDataset(train, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(BertDataset(validation, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(BertDataset(test, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

    # train or load
    if FINETUNE:
        my_trainer(train_loader, val_loader, network, ckpt_path, LAMBDA,
                   LR=LR, epochs=EPOCHS, fresh_training=True)
    else:
        load_model(network, None, PATH=ckpt_path)

    # inference (male / female framing)
    p, _ = get_prediction(test_loader, network, "male")
    test["pred_male_kldm"] = [x[0] for pred in p for x in pred.tolist()]
    p, _ = get_prediction(test_loader, network, "female")
    test["pred_female_kldm"] = [x[0] for pred in p for x in pred.tolist()]

    x_m = int(sum(test["pred_male_kldm"] != test["_original_label"]))
    x_f = int(sum(test["pred_female_kldm"] != test["_original_label"]))
    mf_mismatch = int(sum(test["pred_male_kldm"] != test["pred_female_kldm"]))
    print(f"Male   accuracy: {1 - x_m/len(test):.4f}")
    print(f"Female accuracy: {1 - x_f/len(test):.4f}")
    print(f"Male-Female mismatch: {mf_mismatch}")

    test.to_csv(result_csv, index=False)
    print("Saved predictions ->", result_csv)

    # metrics
    y_true = test["_original_label"].values
    male_pred = test["pred_male_kldm"].values
    female_pred = test["pred_female_kldm"].values

    acc_m, ci_m, ma_m = bootstrap_ci(y_true, male_pred, B=500)
    acc_f, ci_f, ma_f = bootstrap_ci(y_true, female_pred, B=500)
    (spd_mean, spd_lo, spd_hi), (eod_mean, eod_lo, eod_hi) = bootstrap_ci_spd_eod(
        male_pred, female_pred, y_true, n_iterations=500)

    print("--- 95% CI (lower, upper) and 500-run bootstrap mean ---")
    print(f"Accuracy Male   : acc={acc_m*100:.2f}%  95% CI=[{ci_m[0]*100:.2f}, {ci_m[1]*100:.2f}]  mean(500)={ma_m*100:.2f}%")
    print(f"Accuracy Female : acc={acc_f*100:.2f}%  95% CI=[{ci_f[0]*100:.2f}, {ci_f[1]*100:.2f}]  mean(500)={ma_f*100:.2f}%")
    print(f"SPD             : 95% CI=[{spd_lo:.3f}, {spd_hi:.3f}]  mean(500)={spd_mean:.3f}")
    print(f"EOD             : 95% CI=[{eod_lo:.3f}, {eod_hi:.3f}]  mean(500)={eod_mean:.3f}")

    return {
        "task": task_name,
        "lambda": LAMBDA,
        "n_test": len(test),
        "male_acc": acc_m * 100,
        "male_acc_ci_low": ci_m[0] * 100,
        "male_acc_ci_high": ci_m[1] * 100,
        "male_acc_bootmean": ma_m * 100,
        "female_acc": acc_f * 100,
        "female_acc_ci_low": ci_f[0] * 100,
        "female_acc_ci_high": ci_f[1] * 100,
        "female_acc_bootmean": ma_f * 100,
        "male_female_mismatch": mf_mismatch,
        "SPD_mean": spd_mean,
        "SPD_ci_low": spd_lo,
        "SPD_ci_high": spd_hi,
        "EOD_mean": eod_mean,
        "EOD_ci_low": eod_lo,
        "EOD_ci_high": eod_hi,
        "result_csv": result_csv,
    }


## Run all tasks × all LAMBDA values

In [ ]:
all_results = []

for task_name in RUN_TASKS:
    for LAMBDA in LAMBDAS:
        try:
            res = run_experiment(task_name, LAMBDA)
            all_results.append(res)
        except Exception as e:
            print(f"!! FAILED task={task_name} LAMBDA={LAMBDA}: {e}")
            all_results.append({"task": task_name, "lambda": LAMBDA, "error": str(e)})

        # free GPU memory between runs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(all_results)
results_df


## Combined results summary — 95% CI (lower, upper) + 500-run bootstrap mean

For every (task, LAMBDA) run, the table below reports, for **Accuracy (male & female), SPD, and EOD**:
the 95% confidence interval **lower bound**, **upper bound**, and the **mean over the 500 bootstrap runs**.

- Accuracy values are percentages; `acc` is the point estimate on the actual test set and
  `mean` is the average across the 500 bootstrap resamples.
- SPD / EOD are fairness metrics (lower = fairer); `mean` is the 500-run bootstrap mean.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

# Explicit (lower, upper, 500-run mean) columns for every metric.
report_cols = [
    "task", "lambda", "n_test",

    # Accuracy — Male: point acc, CI lower, CI upper, 500-run bootstrap mean
    "male_acc", "male_acc_ci_low", "male_acc_ci_high", "male_acc_bootmean",

    # Accuracy — Female
    "female_acc", "female_acc_ci_low", "female_acc_ci_high", "female_acc_bootmean",

    # SPD: 500-run mean, CI lower, CI upper
    "SPD_mean", "SPD_ci_low", "SPD_ci_high",

    # EOD: 500-run mean, CI lower, CI upper
    "EOD_mean", "EOD_ci_low", "EOD_ci_high",

    "male_female_mismatch",
]
report_cols = [c for c in report_cols if c in results_df.columns]
summary = results_df[report_cols].copy()

# round for display
acc_c = [c for c in report_cols if "acc" in c]
fair_c = [c for c in report_cols if ("SPD" in c or "EOD" in c)]
summary[acc_c] = summary[acc_c].round(2)
summary[fair_c] = summary[fair_c].round(3)
display(summary)

summary.to_csv("all_results_lambda_3_5.csv", index=False)
print("Saved combined summary -> all_results_lambda_3_5.csv")


In [ ]:
# Tidy long-form view: one row per (task, lambda, metric) with lower / upper / mean columns.
rows = []
for _, r in results_df.iterrows():
    if "error" in results_df.columns and pd.notna(r.get("error", np.nan)):
        continue
    rows.append({"task": r["task"], "lambda": r["lambda"], "metric": "Accuracy (Male) %",
                 "point": r["male_acc"], "CI_lower": r["male_acc_ci_low"],
                 "CI_upper": r["male_acc_ci_high"], "mean_500": r["male_acc_bootmean"]})
    rows.append({"task": r["task"], "lambda": r["lambda"], "metric": "Accuracy (Female) %",
                 "point": r["female_acc"], "CI_lower": r["female_acc_ci_low"],
                 "CI_upper": r["female_acc_ci_high"], "mean_500": r["female_acc_bootmean"]})
    rows.append({"task": r["task"], "lambda": r["lambda"], "metric": "SPD",
                 "point": np.nan, "CI_lower": r["SPD_ci_low"],
                 "CI_upper": r["SPD_ci_high"], "mean_500": r["SPD_mean"]})
    rows.append({"task": r["task"], "lambda": r["lambda"], "metric": "EOD",
                 "point": np.nan, "CI_lower": r["EOD_ci_low"],
                 "CI_upper": r["EOD_ci_high"], "mean_500": r["EOD_mean"]})

long_df = pd.DataFrame(rows)
long_df[["point", "CI_lower", "CI_upper", "mean_500"]] = long_df[["point", "CI_lower", "CI_upper", "mean_500"]].round(3)
display(long_df)
long_df.to_csv("all_results_long_lambda_3_5.csv", index=False)
print("Saved long-form summary -> all_results_long_lambda_3_5.csv")


In [ ]:
# Optional: pivoted LAMBDA=3 vs LAMBDA=5 comparison per task (500-run means)
if "male_acc_bootmean" in results_df.columns and (
        "error" not in results_df.columns or results_df["error"].isna().all()):
    for label, metric in [
        ("Accuracy Male %  (500-run mean)", "male_acc_bootmean"),
        ("Accuracy Female % (500-run mean)", "female_acc_bootmean"),
        ("SPD (500-run mean)", "SPD_mean"),
        ("EOD (500-run mean)", "EOD_mean"),
        ("Male-Female mismatch", "male_female_mismatch"),
    ]:
        print(f"\n=== {label} ===")
        display(results_df.pivot(index="task", columns="lambda", values=metric).round(3))
